# Training the small networks of Figures 1 and 4

The note's Figures 1 and 4 show the Jacobian of the parameter-to-output map at random weights. This notebook trains the
same networks on classification datasets and stores them; `analyze.ipynb` then loads them and computes their Jacobians.

**Networks.** As in `experiments.ipynb`, section 10: an embedding $\mathbb R^p \to \mathbb R^4$, $L = 3$ MLP blocks of
width $8$ and a readout $\mathbb R^4 \to \mathbb R^q$, without biases, so $d_{\mathrm{model}} = 4$. Here $p$ is the number
of features and $q \ge 4$ the number of classes, and the readout gives the logits. Activations: $\tanh$, GELU, ReGLU and
SwiGLU. Normalization: none, or pre-RMSNorm with $\varepsilon = 10^{-5}$ before every block and the readout. Three kinds
of network: with identity skips, without skips, and without skips with hidden units added (the fewest whole units that
add at least $L\dim G$ parameters, as in Figure 4).

**Datasets.** Small classification datasets with at least $4$ classes, chosen so that the trained networks beat a tuned
linear classifier. Two are generated by scikit-learn: `hypercube`, $4$ classes of two Gaussian clusters each at vertices
of a $4$-cube, with $2$ noise features ($p = 6$), and `clusters`, $5$ classes of three clusters each in $8$ features. Two
come from the Penn Machine Learning Benchmarks (PMLB) and are downloaded once into `data/`: `satimage`, $6$ classes of
satellite image pixels ($p = 36$), and `segmentation`, $7$ classes of image regions ($p = 18$ after dropping a constant
feature). Datasets with many features or classes, such as scikit-learn's digits, pendigits or letter, are left out: a
linear classifier beats these networks there, since the stream has only $4$ dimensions.

**Training.** Stratified 60/20/20 split into train, validation and test, features standardized on the training set.
AdamW (weight decay $10^{-4}$) on minibatches of $128$, cross-entropy, gradients clipped to norm $1$, a linear warmup, and the learning rate cut by $0.3$
each time the validation loss stalls for about $600$ steps; training stops at the sixth stall, so the rate has fallen
to below $1\%$ of its start, and keeps the weights with the lowest validation loss. For each dataset, activation,
normalization and kind, the learning rate is chosen from $\{10^{-3}, 3\cdot10^{-3}, 10^{-2}, 3\cdot10^{-2}\}$ by the validation loss of
seed $0$, and seeds $1$ to $4$ are trained at that rate. A seed sets the initial weights, $\mathcal N(0, 1/\text{fan-in})$
as in the figures, and the order of the minibatches; with and without skips the same seed starts from the same weights.

**Checks.** Every network is compared with multinomial logistic regression, with an intercept and the $L_2$ penalty
chosen on the validation set. A network counts as trained when it stopped by converging and its test accuracy exceeds
that of the linear classifier. `analyze.ipynb` analyzes trained networks only.

Each network is stored in `runs/<dataset>/<activation>_<normalization>_<kind>_lr<rate>_s<seed>.pt` with its parameters
in float64, its curves and its accuracies. Stored networks are not retrained, so an interrupted run resumes.

In [ ]:
import json, os, platform, time
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
import numpy as np, torch, sklearn
import small_nets as sn

torch.set_num_threads(1)        # one thread per process; the workers below run in parallel
DATASETS = ["hypercube", "clusters", "satimage", "segmentation"]
ACTS = list(sn.ACTS)                      # tanh, GELU, ReGLU, SwiGLU
NORMS = [None, 1e-5]                      # no normalization; pre-RMSNorm with eps = 1e-5, as in Figures 1 and 4
KINDS = ["skip", "noskip", "wide"]
LRS = [1e-3, 3e-3, 1e-2, 3e-2]
SEEDS = range(5)
WORKERS = os.cpu_count()
sn.RUNS.mkdir(exist_ok=True); sn.RESULTS.mkdir(exist_ok=True)
print(f"Python {platform.python_version()}, PyTorch {torch.__version__}, scikit-learn {sklearn.__version__}, "
      f"{WORKERS} CPUs")

In [ ]:
# the datasets and the linear baseline
baselines = {}
for name in DATASETS:
    ds = sn.load(name)
    (Xtr, ytr), (Xva, _), (Xte, _) = ds["train"], ds["val"], ds["test"]
    lin = sn.linear_baseline(ds)
    baselines[name] = {"p": Xtr.shape[1], "q": int(ytr.max()) + 1, "n_train": len(Xtr), "n_val": len(Xva),
                       "n_test": len(Xte), **lin}
    print(f"{name:13s} p = {Xtr.shape[1]:2d}, q = {int(ytr.max()) + 1:2d}, {len(Xtr)}/{len(Xva)}/{len(Xte)} inputs; "
          f"logistic regression: C = {lin['C']:g}, validation {lin['val_acc']:.4f}, test {lin['test_acc']:.4f}")
(sn.RESULTS / "baselines.json").write_text(json.dumps(baselines, indent=1))

In [ ]:
def run(jobs):
    # train the jobs on all CPUs, one thread each; stored networks are loaded instead
    t0 = time.time()
    with ProcessPoolExecutor(WORKERS, mp_context=mp.get_context("fork")) as ex:
        out = list(ex.map(sn.train_and_store, jobs, chunksize=1))
    print(f"{len(jobs)} networks in {time.time() - t0:.0f}s")
    return out

configs = [(ds, act, eps, kind) for ds in DATASETS for act in ACTS for eps in NORMS for kind in KINDS]
# the learning rate: lowest validation loss of seed 0
search = run([(ds, act, eps, kind, lr, 0) for ds, act, eps, kind in configs for lr in LRS])
best_lr = {}
for c in configs:
    rs = [r for r in search if (r["dataset"], r["act"], r["eps"], r["kind"]) == c]
    best_lr[c] = min(rs, key=lambda r: r["val_loss"])["lr"]

In [ ]:
# the other seeds at the chosen rate
final = run([(*c, best_lr[c], seed) for c in configs for seed in SEEDS])
summary = [dict(r, trained=bool(r["stop"] == "converged" and r["test_acc"] > baselines[r["dataset"]]["test_acc"]))
           for r in final]
(sn.RESULTS / "train_summary.json").write_text(json.dumps(summary, indent=1))
seconds = sum(r["seconds"] for r in search) + sum(r["seconds"] for r in final if r["seed"] > 0)
print(f"training time, summed over the networks: {seconds / 60:.1f} min")

In [ ]:
# the check: test accuracy against the linear classifier, and how training stopped
print("test accuracy, median over the seeds (lowest); 'trained' counts the seeds that converged and beat the linear "
      "classifier")
for name in DATASETS:
    print(f"\n{name}: logistic regression {baselines[name]['test_acc']:.4f}")
    for act in ACTS:
        for eps in NORMS:
            row = []
            for kind in KINDS:
                rs = [r for r in summary if (r["dataset"], r["act"], r["eps"], r["kind"]) == (name, act, eps, kind)]
                acc = np.array([r["test_acc"] for r in rs])
                row.append(f"{kind} {np.median(acc):.4f} ({acc.min():.4f}) lr {rs[0]['lr']:.0e} "
                           f"trained {sum(r['trained'] for r in rs)}/{len(rs)}")
            norm = "none" if eps is None else "RMSNorm"
            print(f"  {act:6s} {norm:7s} | " + " | ".join(row))
bad = [r for r in summary if not r["trained"]]
print(f"\n{len(summary) - len(bad)}/{len(summary)} networks trained; stopped by: "
      f"{ {s: sum(r['stop'] == s for r in summary) for s in ('converged', 'max_steps', 'diverged')} }")
for r in bad:
    print("  not trained:", r["dataset"], r["act"], r["eps"], r["kind"], r["seed"], r["stop"], f"{r['test_acc']:.4f}")